# Lebanon Tourism Explorer — Google Colab workflow
Run the six code cells in order. Upload only your tourism CSV in Cell 2. Cell 5 downloads the project files; Cell 6 supplies a temporary testing URL. Deployment and personal review remain your steps.


## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "streamlit==1.55.0", "pandas==2.3.3", "plotly==6.6.0"], check=True)


## Cell 2 — Upload and inspect the original dataset

In [ ]:
from pathlib import Path
from google.colab import files
import pandas as pd

uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_names) != 1:
    raise ValueError("Upload exactly one CSV: the supplied Lebanon tourism dataset.")
project = Path("/content/lebanon-tourism-explorer")
project.mkdir(exist_ok=True)
(project / "lebanon_tourism_data.csv").write_bytes(uploaded[csv_names[0]])
df = pd.read_csv(project / "lebanon_tourism_data.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.dtypes)
display(df.head())
print("Missing values:", df.isna().sum().to_dict())


## Cell 3 — Write the full Streamlit application
The entire application is below, in ordinary Python order. `%%writefile` saves it for Streamlit.

In [ ]:
%%writefile /content/lebanon-tourism-explorer/app.py
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st


# Page setup
st.set_page_config(page_title="Lebanon Tourism Explorer", page_icon="🇱🇧", layout="wide")
st.title("🇱🇧 Lebanon Tourism Explorer")
st.write(
    "Explore where tourism establishments are concentrated in the supplied Lebanon "
    "dataset. Compare governorate profiles, then narrow the town ranking by district. "
    "Each row represents a town and records its Tourism Index and numbers of cafés, "
    "restaurants, hotels, and guest houses."
)

# Load data. Keep the supplied totals and original town names.
data_path = Path(__file__).parent / "lebanon_tourism_data.csv"
try:
    df = pd.read_csv(data_path)
except (OSError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
    st.error(f"Could not read lebanon_tourism_data.csv. Place it beside app.py. Details: {error}")
    st.stop()

metrics = ["tourism_index", "cafes", "restaurants", "hotels", "guest_houses"]
metric_labels = ["Tourism Index", "Cafés", "Restaurants", "Hotels", "Guest houses"]
numeric_columns = metrics + ["total_establishments"]
required_columns = ["town", "district", "governorate"] + numeric_columns
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    st.error("The CSV is missing required columns: " + ", ".join(missing_columns))
    st.stop()
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
if df.empty or df[required_columns].isna().any().any():
    st.error("The dataset is empty or contains missing/invalid required values. Check the CSV; unknown values are not treated as zero.")
    st.stop()

# Full-dataset reference: calculated BEFORE either filter is applied.
# Preserve the original notebook's mean-per-town and min–max normalization.
governorate_averages = df.groupby("governorate")[metrics].mean()
metric_min = governorate_averages.min()
metric_ranges = (governorate_averages.max() - metric_min).replace(0, 1)
normalized = (governorate_averages - metric_min).div(metric_ranges).mul(100)
all_governorates = sorted(governorate_averages.index.tolist())
default_governorates = ["Akkar", "Baalbek-Hermel", "Nabatieh", "North"]

# Interactions: Governorate → District → Town
st.sidebar.header("Explore by area")
selected_governorates = st.sidebar.multiselect(
    "1. Governorates to compare",
    options=all_governorates,
    default=[g for g in default_governorates if g in all_governorates],
    key="governorates",
    help="Updates the radar chart and the available districts below.",
)
governorate_rows = df[df["governorate"].isin(selected_governorates)]
district_options = sorted(governorate_rows["district"].unique().tolist())

# Remove invalid selections BEFORE creating the dependent district widget.
previous_districts = st.session_state.get("districts", [])
st.session_state["districts"] = [d for d in previous_districts if d in district_options]
selected_districts = st.sidebar.multiselect(
    "2. Districts to examine",
    options=district_options,
    key="districts",
    disabled=not selected_governorates,
    placeholder="All available districts",
    help="Leave empty to include all districts in the selected governorates.",
)
st.sidebar.caption("Governorate → District → Town")
st.sidebar.caption("An empty district selection includes all available districts. The radar always compares whole governorates.")

# Filtering
filtered = governorate_rows.copy()
if selected_districts:
    filtered = filtered[filtered["district"].isin(selected_districts)]

if not selected_governorates:
    st.info("Select at least one governorate in the sidebar to explore its towns and tourism profile.")
else:
    st.caption("Town view: " + ", ".join(selected_governorates) + " · " +
               (", ".join(selected_districts) if selected_districts else "All available districts"))
    col1, col2, col3 = st.columns(3)
    col1.metric("Towns in view", f"{len(filtered):,}")
    col2.metric("Establishments in view", f"{filtered['total_establishments'].sum():,.0f}")
    col3.metric("Average town Tourism Index", f"{filtered['tourism_index'].mean():.2f}" if not filtered.empty else "—")

    # Chart 1: Top towns by total establishments
    st.subheader("Where are tourism establishments concentrated?")
    if filtered.empty:
        st.info("No town records match this selection. Choose another district or clear the district filter.")
    else:
        top_towns = filtered.nlargest(12, "total_establishments").sort_values("total_establishments")
        bar = px.bar(
            top_towns,
            x="total_establishments",
            y="town",
            orientation="h",
            color="tourism_index",
            color_continuous_scale="Tealgrn",
            range_color=[df["tourism_index"].min(), df["tourism_index"].max()],
            text="total_establishments",
            labels={"town": "Town", "total_establishments": "Total tourism establishments",
                    "tourism_index": "Tourism Index", "district": "District",
                    "governorate": "Governorate", "cafes": "Cafés", "restaurants": "Restaurants",
                    "hotels": "Hotels", "guest_houses": "Guest houses"},
            hover_data=["district", "governorate", "cafes", "restaurants", "hotels", "guest_houses"],
            title=f"Top {len(top_towns)} towns by number of tourism establishments",
        )
        bar.update_traces(textposition="outside", cliponaxis=False)
        bar.update_layout(template="plotly_white", height=560, margin=dict(l=10, r=40, t=65, b=45))
        bar.update_yaxes(categoryorder="array", categoryarray=top_towns["town"].tolist(), automargin=True)
        bar.update_xaxes(range=[0, max(1, top_towns["total_establishments"].max() * 1.16)])
        st.plotly_chart(bar, width="stretch", key="town_bar")
        st.caption("Counts and bar colors come directly from the CSV. Hover for establishment types and location. Ties at the top-12 cutoff follow CSV row order.")

    # Chart 2: Governorate profiles on a stable common scale
    st.subheader("How do the selected governorates compare?")
    st.caption("Whole-governorate averages per town. District selections affect the town view only.")
    radar = go.Figure()
    palette = px.colors.qualitative.Plotly
    for governorate in selected_governorates:
        values = normalized.loc[governorate].tolist()
        raw_values = governorate_averages.loc[governorate].tolist()
        radar.add_trace(go.Scatterpolar(
            r=values + [values[0]],
            theta=metric_labels + [metric_labels[0]],
            customdata=raw_values + [raw_values[0]],
            fill="toself",
            name=governorate,
            opacity=0.55,
            line=dict(color=palette[all_governorates.index(governorate) % len(palette)]),
            hovertemplate="%{theta}<br>Normalized: %{r:.1f}/100<br>Raw mean per town: %{customdata:.2f}<extra>%{fullData.name}</extra>",
        ))
    radar.update_layout(
        title="Selected governorates have different tourism profiles",
        template="plotly_white",
        height=570,
        showlegend=True,
        polar=dict(radialaxis=dict(visible=True, range=[0, 100], dtick=25)),
        legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center"),
        margin=dict(l=65, r=65, t=75, b=90),
    )
    st.plotly_chart(radar, width="stretch", key="governorate_radar")
    st.caption(
        "For each measure: 100 × (governorate mean − lowest governorate mean) ÷ "
        "(highest − lowest governorate mean), using all seven governorates in the CSV. "
        "0 means the lowest average, not necessarily zero establishments; 100 means the highest. "
        "A measure identical across all governorates is shown as 0. The reference stays fixed when filters change."
    )

    # Insights: compute from the current selection, with explicit scope.
    st.subheader("Key insights")
    if not filtered.empty:
        largest = filtered["total_establishments"].max()
        leaders = filtered[filtered["total_establishments"] == largest]
        leader_names = "; ".join(leaders["town"] + " (" + leaders["district"] + ")")
        st.markdown(f"**Town concentration:** The highest town count is **{largest:,.0f} establishments**, recorded in {leader_names}.")
        district_totals = filtered.groupby("district")["total_establishments"].sum()
        strongest = district_totals[district_totals == district_totals.max()]
        st.markdown(f"**District concentration:** The highest district total is **{strongest.iloc[0]:,.0f} establishments**, recorded in {', '.join(strongest.index)}. Districts with more recorded towns can rank higher because these are totals.")
    selected_means = governorate_averages.loc[selected_governorates, "tourism_index"]
    highest = selected_means[selected_means == selected_means.max()]
    st.markdown(f"**Governorate profile:** {', '.join(highest.index)} has the highest average Tourism Index among the selected governorates: **{highest.iloc[0]:.2f}**. This uses all recorded towns in each selected governorate.")

# Design justification remains visible even when no governorate is selected.
with st.expander("Why these interaction choices?"):
    st.markdown("**Interaction 1 – Governorate Multiselect**")
    st.write(
        'This control answers, “Which governorates do I want to compare?” A multiselect '
        "lets the reader place several profiles on the same radar chart. A single dropdown "
        "would require remembering one profile while viewing another. Choosing a smaller "
        "set focuses attention and reduces overlapping shapes, while the fixed scale keeps "
        "the wider dataset as context. The same choice also controls which districts are available."
    )
    st.markdown("**Interaction 2 – District Multiselect**")
    st.write(
        'This control answers, “Within the selected governorates, which districts do I want '
        'to examine?” A multiselect supports comparing towns across one or several districts. '
        "Showing every district in Lebanon would add irrelevant choices and could create "
        "confusing combinations. Limiting the options to the selected governorates creates "
        "a Governorate → District → Town drill-down, reduces clutter, and focuses attention "
        "on a meaningful subset. Leaving it empty gives an overview of all available districts."
    )

# Data source and interpretation
st.caption(
    "Source: the supplied lebanon_tourism_data.csv from the original Plotly assignment; "
    "its observation column contains AUB-linked CODEC Tourism identifiers. "
    "Coverage: 1,137 town records, 26 districts, 7 governorates; Beirut is not included. "
    "The CSV provides no observation date or Tourism Index formula. Counts describe the supplied "
    "records, not visitor numbers or a current tourism census."
)


## Cell 4 — Create requirements and documentation

In [ ]:
from pathlib import Path
project = Path("/content/lebanon-tourism-explorer")
(project / 'requirements.txt').write_text('streamlit==1.55.0\npandas==2.3.3\nplotly==6.6.0\n', encoding="utf-8")
(project / 'README.md').write_text('# Lebanon Tourism Explorer\n\nA single-page Streamlit assignment exploring tourism establishments in Lebanese towns. It rebuilds two charts from the supplied Plotly assignment as interactive Plotly figures.\n\nPublic Streamlit app: **[Insert public URL after deployment]**\n\n## Visualizations and linked interactions\n\n1. **Top towns:** up to 12 towns by `total_establishments`, shown as horizontal bars colored by `tourism_index` using the original Tealgrn palette. Hover shows district, governorate, index, the four establishment types, and total.\n2. **Governorate profiles:** filled radar polygons for Tourism Index, Cafés, Restaurants, Hotels, and Guest houses. Each metric is the mean across a governorate\'s recorded towns.\n\nThe governorate multiselect changes both the radar traces and the district options. The district multiselect then filters the town ranking, metrics, and town/district insights. This is a **Governorate → District → Town** drill-down. The radar retains whole-governorate context.\n\nAn empty district selection means all districts in the selected governorates. Invalid district selections are removed before the district widget is rendered. Clearing all governorates shows guidance and disables districts.\n\nThe original radar defaults are preserved: Akkar, Baalbek-Hermel, Nabatieh, and North. Three dynamic insights identify the leading town, district, and governorate average. An expander explains the interaction choices.\n\n## Dataset and normalization\n\n`lebanon_tourism_data.csv` is copied unchanged from the supplied assignment. It has 1,137 town records, 26 districts, and 7 governorates. Beirut is absent. Fields include location, Tourism Index, cafés, restaurants, hotels, guest houses, total establishments, attraction flag, tourism group, and an observation identifier. Observation identifiers point to AUB-linked CODEC Tourism records; the supplied CSV is the data source used here. No observation date, index formula, or source license was supplied.\n\nInspection found no missing cells or duplicate town/district/governorate records. All supplied totals equal the sum of the four establishment types. Numeric fields are converted with `pd.to_numeric(errors="coerce")`; invalid required values produce a clear message instead of being silently replaced with zeros. Original totals and names are retained.\n\nThe original notebook and radar HTML agree on this method:\n\n```text\ngovernorate mean = mean of each metric across recorded towns\nnormalized score = 100 × (governorate mean − full-data minimum mean)\n                         / (full-data maximum mean − full-data minimum mean)\n```\n\nMinima and maxima are calculated across **all seven governorates before filtering**, separately for each metric. A constant metric is assigned zero by replacing a zero denominator with one. A normalized score of zero means the lowest governorate mean; it does not necessarily mean no establishments. Radar hover includes the unscaled means. The bar color scale is also fixed to the dataset\'s 0–10 index range.\n\nEstablishment counts are not visitor totals or population-adjusted tourism intensity. District totals can be higher because more towns are recorded. Ranking ties at the 12-town cutoff retain CSV order.\n\n## Run locally\n\nUse Python 3.11 (the tested version) in a terminal opened in this folder:\n\n```bash\npython -m venv .venv\n# Windows PowerShell:\n.venv\\Scripts\\Activate.ps1\n# macOS/Linux instead: source .venv/bin/activate\npython -m pip install -r requirements.txt\npython -m streamlit run app.py\n```\n\nKeep `app.py`, `requirements.txt`, and the CSV together. The CSV path is resolved relative to the app, so launch location does not affect loading.\n\n## Reproduce in Google Colab\n\nUpload `Colab_Workflow.ipynb` into Google Colab and execute its six code cells in order:\n\n1. Install the pinned dependencies.\n2. Upload the original CSV (names with `(1)` are accepted), inspect it, and save the canonical filename.\n3. Write the complete `app.py`.\n4. Write `requirements.txt`, `README.md`, and `.gitignore`.\n5. Compile the app, check the data and files, and exercise linked filters with Streamlit AppTest. Download the GitHub-ready ZIP when prompted.\n6. Start Streamlit with `python -m streamlit run app.py` through the runtime\'s Python executable and open a Cloudflare Quick Tunnel. The cell prints a clickable temporary URL. Keep the runtime running.\n\nThe temporary URL is for testing; it is not the permanent submission link. The Colab tunnel requires outbound network access and downloads Cloudflare\'s official Linux binary. This workflow was prepared and its Python source validated locally; an actual Google Colab runtime and external tunnel still need your live test.\n\n## Push to GitHub\n\nCreate an empty repository on GitHub. Upload these files to its root using **Add file → Upload files**, or run the commands below in this folder (replace the example URL):\n\n```bash\ngit init\ngit add app.py requirements.txt lebanon_tourism_data.csv README.md .gitignore verify_app.py\ngit commit -m "Add Lebanon tourism Streamlit assignment"\ngit branch -M main\ngit remote add origin https://github.com/YOUR-USERNAME/lebanon-tourism-explorer.git\ngit push -u origin main\n```\n\nYou may also upload `Colab_Workflow.ipynb` for reproducibility. No repository has been created or pushed automatically. Keep the submission report outside the public repository unless your instructor requests it.\n\n## Deploy to Streamlit Community Cloud\n\nSign in at [Streamlit Community Cloud](https://share.streamlit.io/), connect GitHub, and choose **Create app**. Select your repository, `main` branch, and `app.py` as the main file. Under advanced settings select Python 3.11, then deploy. Dependencies are read from `requirements.txt`. Copy the actual public URL into this README and the Moodle report after deployment, test both filters, and capture the deployed page.\n\nReferences: [Streamlit deployment guide](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/deploy), [file organization](https://docs.streamlit.io/deploy/streamlit-community-cloud/deploy-your-app/file-organization), [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/).\n\n## Verification\n\n```bash\npython verify_app.py\n```\n\nThis checks compilation, data integrity, default charts, dependent district options, district filtering, removal of stale selections, empty selections, radar trace updates, and stable normalization. The original delivered project also includes `Verification.md` with local development results.\n\nManual demonstration: select Mount Lebanon, inspect its available districts, choose Baabda, then clear districts. Next select North and compare the radar. Finally clear all governorates and confirm the guidance message.\n\n## Submission and AI disclosure\n\nUse the separate `Moodle_Report.txt` as paste-ready report text. Fill in your name, real links, screenshot, and only the personal review/testing actions you actually complete. Codex/ChatGPT assisted with code, charts, filters, testing, troubleshooting, and documentation. No course AI-policy document was supplied; adapt the disclosure to your instructor\'s requirements.\n', encoding="utf-8")
(project / '.gitignore').write_text('__pycache__/\n*.pyc\n.venv/\n.ipynb_checkpoints/\n.streamlit/secrets.toml\n*.log\ncloudflared\n', encoding="utf-8")
print("Created requirements.txt, README.md, and .gitignore")


## Cell 5 — Validate and download the GitHub-ready files
Checks syntax, data, and actual Streamlit widget behavior. A ZIP download follows successful checks. Keep your original workflow notebook separately; Colab does not automatically copy an open notebook into its runtime folder.

In [ ]:
import py_compile, subprocess, sys, zipfile
import pandas as pd
from google.colab import files

py_compile.compile(str(project / "app.py"), doraise=True)
df = pd.read_csv(project / "lebanon_tourism_data.csv")
assert len(df) == 1137, "Check that you uploaded the supplied dataset."
for name in ["app.py", "requirements.txt", "README.md", "lebanon_tourism_data.csv"]:
    assert (project / name).is_file(), name
(project / "verify_app.py").write_text('"""Run with python verify_app.py; no extra test packages are required."""\nimport json\nfrom pathlib import Path\nimport py_compile\n\nimport pandas as pd\nfrom streamlit.testing.v1 import AppTest\n\nROOT = Path(__file__).parent\npy_compile.compile(str(ROOT / "app.py"), doraise=True)\ndf = pd.read_csv(ROOT / "lebanon_tourism_data.csv")\nassert len(df) == 1137\nassert not df.isna().any().any()\nassert not df.duplicated(["town", "district", "governorate"]).any()\nassert df["total_establishments"].equals(df[["cafes", "restaurants", "hotels", "guest_houses"]].sum(axis=1))\nfor name in ["app.py", "requirements.txt", "README.md", "lebanon_tourism_data.csv"]:\n    assert (ROOT / name).is_file(), name\n\n\ndef charts(app):\n    return [json.loads(chart.proto.spec) for chart in app.get("plotly_chart")]\n\n\napp = AppTest.from_file(str(ROOT / "app.py"), default_timeout=30).run()\nassert not app.exception, app.exception\nassert app.multiselect[0].value == ["Akkar", "Baalbek-Hermel", "Nabatieh", "North"]\nassert len(charts(app)) == 2\noriginal_radar = {trace["name"]: trace["r"] for trace in charts(app)[1]["data"]}\nassert len(original_radar) == 4\nassert any(e.label == "Why these interaction choices?" for e in app.expander)\nassert sum("**" in m.value for m in app.markdown) >= 5\n\napp.multiselect[0].set_value(["Mount Lebanon"]).run()\nassert not app.exception, app.exception\nexpected_options = sorted(df.loc[df.governorate == "Mount Lebanon", "district"].unique())\nassert app.multiselect[1].options == expected_options\nassert [t["name"] for t in charts(app)[1]["data"]] == ["Mount Lebanon"]\nbefore = charts(app)[0]\napp.multiselect[1].set_value(["Baabda"]).run()\nassert not app.exception, app.exception\nafter = charts(app)[0]\nassert before != after\nassert all(row[0] == "Baabda" for row in after["data"][0]["customdata"])\nexpected_top = df[(df.governorate == "Mount Lebanon") & (df.district == "Baabda")].nlargest(12, "total_establishments").sort_values("total_establishments")\nassert list(after["data"][0]["y"]) == expected_top.town.tolist()\nassert app.metric[0].value == str(len(df[df.district == "Baabda"]))\n\n# Retain valid selections when another governorate is added.\napp.multiselect[0].set_value(["Mount Lebanon", "North"]).run()\nassert app.multiselect[1].value == ["Baabda"]\n# Remove stale districts when their governorate is removed.\napp.multiselect[0].set_value(["North"]).run()\nassert not app.exception, app.exception\nassert app.multiselect[1].value == []\nassert app.metric[0].value == str(len(df[df.governorate == "North"]))\nassert charts(app)[1]["data"][0]["r"] == original_radar["North"]\n\napp.multiselect[0].set_value([]).run()\nassert not app.exception, app.exception\nassert not app.multiselect[1].options and app.multiselect[1].disabled\nassert not charts(app)\nassert any("Select at least one governorate" in i.value for i in app.info)\nassert len(app.expander) == 1\n\napp.multiselect[0].set_value(sorted(df.governorate.unique())).run()\nassert not app.exception, app.exception\nassert len(charts(app)[1]["data"]) == 7\nmeans = df.groupby("governorate")[["tourism_index", "cafes", "restaurants", "hotels", "guest_houses"]].mean()\nexpected_normalized = 100 * (means - means.min()) / (means.max() - means.min()).replace(0, 1)\nfor trace in charts(app)[1]["data"]:\n    expected = expected_normalized.loc[trace["name"]].tolist()\n    assert all(abs(a - b) < 1e-10 for a, b in zip(trace["r"], expected + [expected[0]]))\n\nprint("PASS: compilation, CSV integrity, required files, charts, linked filters, stale-state cleanup, empty selections, insights, justification, and full-data radar normalization.")\n', encoding="utf-8")
subprocess.run([sys.executable, str(project / "verify_app.py")], check=True)
archive = Path("/content/lebanon-tourism-explorer.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["app.py", "requirements.txt", "README.md", "lebanon_tourism_data.csv", ".gitignore", "verify_app.py"]:
        z.write(project / name, arcname=name)
files.download(str(archive))
print("Checks passed. The downloaded ZIP contains the GitHub-ready app.")


## Cell 6 — Run Streamlit and create a temporary Cloudflare URL
Keep this runtime alive while testing. Rerunning the cell stops only the processes previously started by this cell. The public temporary link expires when the tunnel stops; use Streamlit Community Cloud for your submission link.

In [ ]:
import os, platform, re, subprocess, sys, time, urllib.request
from IPython.display import display, HTML

# Stop only our previous launch when rerunning this cell.
for name in ["tunnel_process", "streamlit_process"]:
    old_process = globals().get(name)
    if old_process is not None and old_process.poll() is None:
        old_process.terminate()
        try:
            old_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            old_process.kill()
            old_process.wait()

binary = Path("/content/cloudflared")
architecture = "arm64" if platform.machine() in ("aarch64", "arm64") else "amd64"
if not binary.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-" + architecture,
        binary,
    )
binary.chmod(0o755)
streamlit_log = Path("/content/streamlit.log")
tunnel_log = Path("/content/cloudflared.log")
with streamlit_log.open("w") as log:
    streamlit_process = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", "app.py",
         "--server.address=0.0.0.0", "--server.port=8501",
         "--server.headless=true", "--browser.gatherUsageStats=false"],
        cwd=project, stdout=log, stderr=subprocess.STDOUT,
    )

for attempt in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError(streamlit_log.read_text())
    try:
        with urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2) as response:
            if response.status == 200:
                break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not become ready. " + streamlit_log.read_text())

with tunnel_log.open("w") as log:
    tunnel_process = subprocess.Popen(
        [str(binary), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT,
    )
for attempt in range(90):
    log_text = tunnel_log.read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        url = match.group(0)
        display(HTML(f'<p><a href="{url}" target="_blank">Open Lebanon Tourism Explorer</a></p>'))
        print("Temporary URL:", url)
        print("Allow a few seconds for the tunnel, then test both filters. Keep the runtime running.")
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError("Cloudflare tunnel stopped: " + log_text)
    time.sleep(1)
else:
    raise RuntimeError("No tunnel URL received. Inspect /content/cloudflared.log and rerun Cell 6. " + log_text[-2000:])
